In [24]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# Image processing libraries
import nibabel as nib
import scipy.ndimage as ndimage

# Scikit-learn for preprocessing
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# TensorFlow / Keras for Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class MultimodalDataGenerator(tf.keras.utils.Sequence):
    """
    Custom generator to load TWO 3D CT scans (volume + binary mask) and tabular data.
    """
    def __init__(self, df, tabular_feature_cols, target_col, batch_size=8, target_shape=(32, 64, 64), is_training=True):
        self.df = df.reset_index(drop=True)
        self.tabular_cols = tabular_feature_cols
        self.target_col = target_col
        self.batch_size = batch_size
        self.target_shape = target_shape
        self.is_training = is_training

    def __len__(self):
        return int(np.ceil(len(self.df) / float(self.batch_size)))

    def _resize_volume(self, img_array, is_mask=False):
        """Resizes a 3D volume to the target shape."""
        factors = (
            self.target_shape[0] / img_array.shape[0],
            self.target_shape[1] / img_array.shape[1],
            self.target_shape[2] / img_array.shape[2]
        )

        # order=0 (Nearest Neighbor) for binary masks to preserve exact 0s and 1s
        # order=1 (Spline) for the continuous CT volume
        order = 0 if is_mask else 1
        resized = ndimage.zoom(img_array, factors, order=order)

        if not is_mask:
            # Normalize CT voxel values to [0, 1]
            min_val = np.min(resized)
            max_val = np.max(resized)
            if max_val - min_val > 0:
                resized = (resized - min_val) / (max_val - min_val)

        return resized

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]

        batch_images = []
        batch_tabular = []
        batch_targets = []

        for _, row in batch_df.iterrows():
            vol_path = row['volume_path']
            mask_path = row['mask_path']

            try:
                # Load both volume and mask
                vol_data = nib.load(vol_path).get_fdata()
                mask_data = nib.load(mask_path).get_fdata()

                # Resize both to the same dimensions
                vol_resized = self._resize_volume(vol_data, is_mask=False)
                mask_resized = self._resize_volume(mask_data, is_mask=True)

                # Stack them into a 2-channel array: Shape (Depth, Height, Width, 2)
                stacked_img = np.stack([vol_resized, mask_resized], axis=-1)

            except Exception as e:
                print(f"Failed to load images for patient {row.get('patient_id', 'Unknown')}: {e}")
                # Fallback to zeros (2 channels) on error to prevent batch crash
                stacked_img = np.zeros((*self.target_shape, 2))

            batch_images.append(stacked_img)
            batch_tabular.append(row[self.tabular_cols].values.astype(np.float32))

            if self.target_col in row:
                batch_targets.append(row[self.target_col])

        X_img = np.array(batch_images, dtype=np.float32)
        X_tab = np.array(batch_tabular, dtype=np.float32)
        y = np.array(batch_targets, dtype=np.float32)

        if self.is_training:
            return ({"image_input": X_img, "tabular_input": X_tab}, y)
        return {"image_input": X_img, "tabular_input": X_tab}


class MultimodalHealthcareClassifier:
    def __init__(self, target_column: str = 'malignancy_score', patient_id_column: str = 'patient_id'):
        self.target_column = target_column
        self.patient_id_column = patient_id_column
        self.data: pd.DataFrame = None
        self.scaler: StandardScaler = None
        self.imputer: SimpleImputer = None
        self.label_encoder: LabelEncoder = None
        self.model = None
        self.tabular_features = []
        self.num_classes = 0

    def load_data(self, tabular_data_path: str, image_root_directory: str) -> None:
        """
        Reads the central Excel/CSV, then pairs the preprocessed_volume and tumor_mask.
        """
        print("Reading central tabular data...")
        if tabular_data_path.endswith('.xlsx') or tabular_data_path.endswith('.xls'):
            self.data = pd.read_excel(tabular_data_path)
        else:
            self.data = pd.read_csv(tabular_data_path)

        root_path = Path(image_root_directory)

        vol_paths = []
        mask_paths = []

        print("Locating NIfTI images in patient folders...")
        for _, row in self.data.iterrows():
            patient_id = str(row[self.patient_id_column])
            patient_folder = root_path / patient_id

            v_path = None
            m_path = None

            if patient_folder.exists():
                # Find Volume and Mask
                vol_file = list(patient_folder.rglob("*preprocessed_volume*.nii.gz"))
                mask_file = list(patient_folder.rglob("*tumor_mask*.nii.gz"))

                if vol_file: v_path = str(vol_file[0])
                if mask_file: m_path = str(mask_file[0])

            vol_paths.append(v_path)
            mask_paths.append(m_path)

        # Add the extracted paths to our dataframe
        self.data['volume_path'] = vol_paths
        self.data['mask_path'] = mask_paths

        # Drop rows that are missing the target or EITHER of the image files
        initial_len = len(self.data)
        self.data = self.data.dropna(subset=[self.target_column, 'volume_path', 'mask_path'])
        dropped = initial_len - len(self.data)

        print(f"Loaded {len(self.data)} complete multimodal records (Dropped {dropped} incomplete rows).")

    def preprocess_tabular_data(self) -> None:
        """Scales and imputes tabular features and dynamically encodes target."""
        # Encode Target (e.g., Scores 1,2,3,4,5 become 0,1,2,3,4)
        self.label_encoder = LabelEncoder()
        self.data[self.target_column] = self.label_encoder.fit_transform(self.data[self.target_column])

        # Record number of classes for the network build step
        self.num_classes = len(self.label_encoder.classes_)
        print(f"Target variable encoded. Detected {self.num_classes} unique classes.")

        # Define tabular features (exclude metadata and target)
        exclude = [self.patient_id_column, self.target_column, 'volume_path', 'mask_path']
        self.tabular_features = [col for col in self.data.columns if col not in exclude and pd.api.types.is_numeric_dtype(self.data[col])]

        self.imputer = SimpleImputer(strategy='mean')
        self.scaler = StandardScaler()

        self.data[self.tabular_features] = self.imputer.fit_transform(self.data[self.tabular_features])
        self.data[self.tabular_features] = self.scaler.fit_transform(self.data[self.tabular_features])
        print(f"Preprocessed {len(self.tabular_features)} tabular features.")

    def build_multimodal_network(self, image_shape=(32, 64, 64, 2)):
        """Builds a dual-input Keras model configured for Multi-Class Classification."""
        num_tabular_features = len(self.tabular_features)

        # --- BRANCH 1: 3D Image Processor (2 Channels: CT + Mask) ---
        img_input = layers.Input(shape=image_shape, name="image_input")

        x = layers.Conv3D(32, kernel_size=3, activation="relu", padding="same")(img_input)
        x = layers.MaxPooling3D(pool_size=2)(x)
        x = layers.BatchNormalization()(x)

        x = layers.Conv3D(64, kernel_size=3, activation="relu", padding="same")(x)
        x = layers.MaxPooling3D(pool_size=2)(x)
        x = layers.BatchNormalization()(x)

        x = layers.GlobalAveragePooling3D()(x)
        img_features = layers.Dense(64, activation="relu")(x)

        # --- BRANCH 2: Tabular Data Processor ---
        tab_input = layers.Input(shape=(num_tabular_features,), name="tabular_input")
        y = layers.Dense(64, activation="relu")(tab_input)
        y = layers.Dropout(0.2)(y)
        tab_features = layers.Dense(32, activation="relu")(y)

        # --- FUSION: Combine Both Branches ---
        combined = layers.Concatenate()([img_features, tab_features])
        z = layers.Dense(64, activation="relu")(combined)
        z = layers.Dropout(0.3)(z)

        # Output Layer (Softmax for Multi-class)
        output = layers.Dense(self.num_classes, activation="softmax", name="prediction")(z)

        self.model = keras.Model(inputs=[img_input, tab_input], outputs=output)

        # Compile with Sparse Categorical Crossentropy for integer labels
        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-4),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        print("Model compiled successfully for Multi-Class Classification. Architecture:")
        print(self.model.summary())

    def split_and_train(self, epochs=20, batch_size=8, image_shape=(32, 64, 64)):
        """Splits patients, initializes generators, and trains the model."""
        groups = self.data[self.patient_id_column]
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, val_idx = next(gss.split(self.data, groups=groups))

        train_df = self.data.iloc[train_idx]
        val_df = self.data.iloc[val_idx]

        print(f"Training on {len(train_df)} scans, Validating on {len(val_df)} scans.")

        train_gen = MultimodalDataGenerator(train_df, self.tabular_features, self.target_column, batch_size, image_shape)
        val_gen = MultimodalDataGenerator(val_df, self.tabular_features, self.target_column, batch_size, image_shape)

        print("\nStarting Multimodal Training...")

        early_stopping = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )

        history = self.model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=epochs,
            callbacks=[early_stopping]
        )
        return history

# ==========================================
# HOW TO RUN IT
# ==========================================

if __name__ == "__main__":
    # 1. Define paths
    EXCEL_FILE_PATH = r"C:\path\to\your\central_data.xlsx"
    IMAGE_ROOT_DIR = r"C:\path\to\LIDC-IDRI\DATA WE WORK ON"

    # 2. Set targets - Make sure TARGET_VAR matches your Excel header exactly
    TARGET_VAR = "malignancy_score"
    PATIENT_ID_VAR = "patient_id"

    pipeline = MultimodalHealthcareClassifier(
        target_column=TARGET_VAR,
        patient_id_column=PATIENT_ID_VAR
    )

    try:
        pipeline.load_data(EXCEL_FILE_PATH, IMAGE_ROOT_DIR)
        pipeline.preprocess_tabular_data()
        pipeline.build_multimodal_network(image_shape=(32, 64, 64, 2))
        history = pipeline.split_and_train(epochs=20, batch_size=8, image_shape=(32, 64, 64))
        print("\nPipeline execution complete!")

    except Exception as e:
        print(f"\nAn error occurred during execution: {e}")

Reading central tabular data...

An error occurred during execution: [Errno 2] No such file or directory: 'C:\\path\\to\\your\\central_data.xlsx'
